# QLoRA fine-tuning: ticket-router (Qwen2.5-1.5B-Instruct)

Fine-tunes the open-weights model set in `train/config.yaml` (`base_model`, default
`Qwen/Qwen2.5-1.5B-Instruct`) on the ticket-routing SFT data from step-06, so it can be
self-hosted, quantized, and served in Docker in later steps (09-12). **Not** a hosted-API
fine-tune -- see the project's `README.md` "Next steps" section for why.

**Runs on Google Colab only (free T4 GPU runtime). No training happens on the local machine.**

## Expected runtime (free T4, approximate -- varies with Colab's GPU allocation/throttling)
- Smoke test (~100 steps): **~5-10 minutes** (now also prints a checkpoint disk-size check)
- Full run (3 epochs, ~2,450 steps at effective batch size 16, eval/save every 250 steps
  against a 400-example subsample): **~45-90 minutes**
- Final best-checkpoint selection (`save_total_limit` is null -- EVERY checkpoint from the
  full run survives, ~10 of them at save_steps=250 -- each gets one full-validation pass,
  1,635 examples): **~+15-30 minutes**

## Expected VRAM (free T4 has 16 GB total)
- **~6-10 GB** during training (4-bit NF4 base model + LoRA adapters + activations with
  gradient checkpointing + paged 8-bit optimizer states) -- comfortable headroom on a T4.

## Expected disk (Colab-local or Google Drive, wherever `REPO_ROOT` points)
- LoRA adapter checkpoints only (not full model weights) -- a few tens of MB each. The
  smoke-test cell (step 11) measures one real checkpoint's size and prints the projected
  total for all ~10 full-run checkpoints (`save_total_limit` is null, so none get deleted),
  compared against free disk space at `REPO_ROOT`. If it doesn't fit, the cell prints a
  warning and stops short of recommending a fix -- decide with the user before changing
  `save_total_limit`, rather than silently shrinking it.

## Cell order -- run top to bottom, in this exact order
1. Check GPU (must show a T4 or better -- stop if none) -- also decides bf16 vs. fp16 for this
   session's GPU, printed here before anything else runs
2. Install dependencies
3. Get the repo (git clone for code + Google Drive/upload for the gitignored data files)
4. Load `train/config.yaml`
5. **Check a Hugging Face token with WRITE access is present** -- before any training runs,
   not after a ~45-90 minute run is done and the push step fails
6. Load tokenizer + SFT datasets (full validation set, and a fixed seeded subsample used only
   for speed during mid-training evaluation -- see `train/config.yaml`)
7. **Verify completion-only masking** (prints one full example's per-token labels)
8. **Verify padding is dynamic per-batch, not fixed to `max_seq_len`**
9. Load base model in 4-bit (QLoRA) + attach LoRA
10. Build the trainer factory from `train/config.yaml` (nothing hardcoded here)
11. **Smoke test** (~100 steps) -- loss curve + 5 generated JSON outputs + a checkpoint
    disk-size projection for the full run
12. --- STOP AND CONFIRM before continuing (read the smoke-test results first, INCLUDING the
    disk projection) ---
13. **Full training run** (mid-training eval/loss-curve uses the SUBSAMPLE, for speed;
    `save_total_limit` is null, so every checkpoint is kept)
14. **Select the best checkpoint using the FULL validation set** -- the subsample used during
    training is for speed only and is never used to pick the final checkpoint; every
    checkpoint saved during the run is evaluated here, since none were deleted
15. Save the adapter + push it to the Hugging Face Hub
16. **Post-train preview**: generate + print alongside gold labels on 20 val examples

> **A note on what was and wasn't verified before this notebook was written**: the
> completion-only-masking mechanism (cell 7) and the dynamic-padding behavior (cell 8) were
> verified by hand, token-by-token, against Qwen2.5's real tokenizer before this notebook was
> written -- see `PROGRESS.md`'s step-07 entry for the full worked example. The end-to-end
> QLoRA training loop itself (bitsandbytes 4-bit loading, the actual GPU training step, and the
> new full-validation checkpoint-selection cell) could **not** be executed locally (no GPU
> available, and a local disk-space constraint blocked even a CPU-only `trl`/`torch` install for
> further testing) -- it's written to the current standard `transformers`/`trl`/`peft` API, but
> you should expect to debug minor version-API mismatches on first run. Please report back what
> you see so this can be corrected if needed.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime menu -> Change runtime type -> T4 GPU -> Save, then re-run from the top."
    )

# bf16 needs Ampere-or-newer (CUDA compute capability major >= 8). The free Colab tier usually
# allocates a T4, which is Turing (compute capability 7.5) and does NOT support bf16 -- decide
# this at runtime from the actual allocated GPU rather than hardcoding bfloat16 in config.yaml,
# which would be silently wrong on a T4. COMPUTE_DTYPE/USE_BF16 are used below by both the QLoRA
# BitsAndBytesConfig (cell 9) and the trainer's mixed-precision flag (cell 10).
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"GPU detected: {gpu_name} (compute capability {major}.{minor})")
print(f"bf16 supported: {USE_BF16} -> using compute dtype: {COMPUTE_DTYPE}")

## 2. Install dependencies

Training-specific packages, installed only inside this Colab runtime -- not part of the
project's own `requirements.txt` (that file is for the local data/harness pipeline, which
never needs a GPU training stack).

In [ ]:
# Exact versions confirmed by probing a real Colab session (not assumed) -- ALL pinned now
# except torch: transformers==5.17.0, trl==1.13.0, peft==0.20.0, accelerate==1.14.0,
# bitsandbytes==0.50.2, datasets==4.8.5, pyyaml==6.0.3, huggingface_hub==1.29.0. Pinned on
# purpose -- an earlier, unpinned version of this cell let trl drift onto 1.13.0 silently,
# which removed trl.DataCollatorForCompletionOnlyLM entirely (see train/config.yaml's
# "Completion-only loss" comment) and broke the notebook in a way that wasn't caught until
# probed by hand. accelerate/bitsandbytes/datasets/pyyaml/huggingface_hub were left unpinned
# in an earlier version of this cell for exactly the same reason -- their versions hadn't
# been read from a real session yet either, and guessing would have repeated the mistake.
#
# torch is the one deliberate exception -- it ships preinstalled with Colab's runtime image
# (observed 2.11.0+cu128 on the session that surfaced the trl break), and pinning it risks
# fighting Colab's own CUDA-matched build. Only trl/transformers/peft's API surface (and,
# it turned out, nothing else) is what broke last time, not torch's.
!pip install -q "transformers==5.17.0" "trl==1.13.0" "peft==0.20.0" "accelerate==1.14.0" "bitsandbytes==0.50.2" "datasets==4.8.5" "pyyaml==6.0.3" "huggingface_hub==1.29.0"

In [ ]:
import importlib.metadata as _im

print("Pinned (confirmed exact versions):")
for _pkg in ["transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "pyyaml", "huggingface_hub"]:
    try:
        print(f"  {_pkg}: {_im.version(_pkg)}")
    except _im.PackageNotFoundError:
        print(f"  {_pkg}: {_im.version('PyYAML' if _pkg == 'pyyaml' else _pkg)}")

print("\ntorch (unpinned, from Colab's own image):")
import torch as _torch_version_check
print(f"  torch: {_torch_version_check.__version__}")

## 3. Get the repo

`data/processed/` (the parquet splits and the SFT `.jsonl` files from step-06) is **gitignored** -- a plain `git clone` will get you the notebook, `train/config.yaml`, and the `src/` package, but **not** the actual training data. Upload the gitignored files into `REPO_ROOT` yourself (Colab's file-browser upload, or the Files > Upload widget) -- `git clone` alone is not enough.

**Default: Google Drive** (`/content/drive/MyDrive/ticket-router`) -- the Colab runtime disconnects frequently when left idle, and a disconnect wipes `/content` entirely. Drive survives that: both the uploaded data and every saved checkpoint persist, so a dropped session costs at most the steps since the last checkpoint, not the whole run. Uploading into `MyDrive/ticket-router` is a one-time setup. The cell below mounts Drive, creates the folders this notebook needs, and reports which required files are still missing.

**Alternative: Colab-local disk** (commented out below) -- faster I/O, but `/content` (and everything in it -- data and checkpoints alike) is wiped on every disconnect. Not the default -- uncomment to switch to it.

In [ ]:
import os

# --- Default: Google Drive (persists across a Colab disconnect) ---------------------------
# The Colab runtime disconnects frequently when left idle. A disconnect wipes /content
# entirely -- both the uploaded training data AND every saved checkpoint would be lost, and
# the run would restart from step 0. Mounting Drive and pointing REPO_ROOT at it means a
# dropped session costs at most the steps since the last checkpoint (every save_steps, per
# train/config.yaml), not the whole run.
#
# The I/O cost of using Drive here is small, and this is why: the JSONL datasets are read
# ONCE by load_dataset() (step 6) -- `datasets` then memory-maps them from its local disk
# cache for every subsequent batch read during training, not from Drive again. Checkpoints
# are LoRA adapters only -- a few tens of MB each (see step 11's disk-size check), written
# roughly 10 times across the full run (save_steps=250 over ~2,450 steps) -- not large or
# frequent enough for Drive's I/O overhead to matter.
#
# The base model's own weights are NOT stored on Drive -- AutoModelForCausalLM.from_pretrained
# (step 9) downloads them to Colab's LOCAL Hugging Face cache every session, on purpose:
# re-downloading ~3GB from the Hub each session is cheaper than reading it back over Drive's
# FUSE mount.
#
# Uploading the repo's gitignored files into MyDrive/ticket-router is a ONE-TIME setup, not
# something redone per session -- once they're in Drive, every future session finds them
# already there.
from google.colab import drive
drive.mount('/content/drive')
REPO_ROOT = "/content/drive/MyDrive/ticket-router"

# --- Alternative: Colab-local disk -- uncomment this line AND comment out/remove the Drive ---
# mount + REPO_ROOT line above to use it instead. Faster I/O, but /content is wiped on every
# Colab disconnect -- both the uploaded data and any in-progress checkpoints would be lost.
# REPO_ROOT = "/content/ticket-router"

# Folders this notebook writes into or reads from -- created if missing, so uploading files
# into them (rather than a git clone, which would create them automatically) doesn't need the
# directory structure to exist by hand first.
os.makedirs(f"{REPO_ROOT}/train", exist_ok=True)
os.makedirs(f"{REPO_ROOT}/data/processed/sft", exist_ok=True)

# Required files -- data/processed/ is gitignored, so a plain `git clone` alone will NOT bring
# these in; upload them separately. A fresh REPO_ROOT legitimately has nothing in it the first
# time this cell runs, so this REPORTS what's still missing instead of asserting on it.
required_files = {
    "train/config.yaml": f"{REPO_ROOT}/train/config.yaml",
    "data/processed/sft/train.jsonl": f"{REPO_ROOT}/data/processed/sft/train.jsonl",
    "data/processed/sft/val.jsonl": f"{REPO_ROOT}/data/processed/sft/val.jsonl",
    "src/ (the ticket_router package)": f"{REPO_ROOT}/src/ticket_router",
}
missing = [label for label, path in required_files.items() if not os.path.exists(path)]

print(f"REPO_ROOT = {REPO_ROOT}")
if missing:
    print("\nStill need to upload before continuing to the next cell:")
    for label in missing:
        print(f"  - {label}")
else:
    print("\nAll required files found -- ready to continue.")

## 4. Load `train/config.yaml`

Every hyperparameter used below comes from this file -- nothing is hardcoded in this notebook.

In [ ]:
import sys, yaml

sys.path.insert(0, f"{REPO_ROOT}/src")  # so `from ticket_router...` works later (schema reuse)

with open(f"{REPO_ROOT}/train/config.yaml") as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, sort_keys=False))

## 5. Check Hugging Face token (write access)

Training pushes the adapter to `hub_model_id` from `train/config.yaml`
(`Omar-Mohsen22/ticket-router-qwen2.5-1.5b-qlora`) at the very end. Checking this **now**, before
the smoke test or the full run, means a missing/insufficient token fails in seconds -- not after
a ~45-90 minute training run, at the push step.

In [ ]:
from huggingface_hub import HfApi, get_token
from huggingface_hub.utils import HfHubHTTPError

token = get_token()  # HF_TOKEN env var, a Colab "HF_TOKEN" secret, or a prior login() call
if not token:
    raise RuntimeError(
        "No Hugging Face token found. This notebook pushes the trained adapter to "
        f"'{config['hub_model_id']}' (train/config.yaml's hub_model_id) and needs a WRITE-access "
        "token before training starts, not after. In Colab: add a secret named HF_TOKEN (key "
        "icon, left sidebar) holding a token with 'Write' access "
        "(https://huggingface.co/settings/tokens), or run `from huggingface_hub import login; "
        "login()` in a cell above and re-run this cell."
    )

api = HfApi(token=token)
try:
    who = api.whoami()
    # Token permission fields vary in shape by token type (classic vs. fine-grained), so the
    # reliable check is a real write probe: create the target repo now if it doesn't exist yet
    # (harmless and idempotent, exist_ok=True) rather than parsing whoami()'s scope structure.
    api.create_repo(config["hub_model_id"], private=config["hub_private"], exist_ok=True)
except HfHubHTTPError as e:
    raise RuntimeError(
        f"Hugging Face token found but does not have WRITE access to "
        f"'{config['hub_model_id']}'. Generate a token with 'Write' access at "
        f"https://huggingface.co/settings/tokens, then set it as the HF_TOKEN Colab secret or "
        f"call login() again.\nOriginal error: {e}"
    ) from e

print(f"HF token OK: logged in as '{who['name']}', write access to '{config['hub_model_id']}' confirmed.")

## 6. Load tokenizer + SFT datasets

Loads the full validation set (used only once, for final best-checkpoint selection in step 14)
and a fixed, seeded subsample of it (`eval_subsample_size`/`eval_subsample_seed` in
`train/config.yaml`) used for speed during mid-training evaluation in steps 11 and 13.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(config["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_path = f"{REPO_ROOT}/{config['train_file']}"
val_path = f"{REPO_ROOT}/{config['val_file']}"

raw_datasets = load_dataset("json", data_files={"train": train_path, "val": val_path})
print(raw_datasets)
print("\nOne raw example -- TRL's prompt-completion format (prompt: system+user, completion: assistant):")
print("prompt:", raw_datasets["train"][0]["prompt"])
print("completion:", raw_datasets["train"][0]["completion"])

# Fixed, seeded subsample of the full validation set -- SPEED ONLY, used for mid-training
# evaluation (steps 11 and 13's loss curves). The FULL validation set (raw_datasets["val"]) is
# kept untouched and is used exactly once, for final best-checkpoint selection in step 14. See
# train/config.yaml's "Evaluation & checkpointing" comments for why these are kept separate.
eval_subsample = raw_datasets["val"].shuffle(seed=config["eval_subsample_seed"]).select(
    range(min(config["eval_subsample_size"], len(raw_datasets["val"])))
)
print(
    f"\nFull validation set: {len(raw_datasets['val'])} examples (final checkpoint selection only)."
    f"\nMid-training eval subsample: {len(eval_subsample)} examples "
    f"(seed={config['eval_subsample_seed']}, speed only -- NOT the metric used to pick the final checkpoint)."
)

## 7. Verify completion-only masking

The loss must be computed **only** on the assistant's JSON response -- prompt tokens (system +
user) contribute nothing to the loss or its gradient. Otherwise most of the model's fine-tuning
capacity goes into learning to *write* support tickets (reproducing the prompt) instead of
learning to *classify* them (producing the completion) -- the actual task.

**Mechanism changed from the original step-07 write-up.** That version used
`trl.DataCollatorForCompletionOnlyLM`, keyed on a `response_template` string. Confirmed against
the real Colab environment (trl 1.13.0, transformers 5.17.0, peft 0.20.0) that this class no
longer exists -- **trl exports no collator classes at all in this version**. The obvious
replacement, `SFTConfig(assistant_only_loss=True)` (masks via the chat template's own
`{% generation %}` markers), was tested and **rejected**: Qwen2.5's chat template has no
`{% generation %}` markers, so `apply_chat_template(return_assistant_tokens_mask=True)` returns
an `assistant_masks` array that is **all zeros** -- it does not raise, it would silently mask
every token and train on nothing.

**Fixed by changing the dataset format, not by finding another collator.**
`data/processed/sft/*.jsonl` now stores `{"prompt": [system, user], "completion": [assistant]}`
per example (TRL's conversational prompt-completion format -- see
`src/ticket_router/make_sft_dataset.py`), and `train/config.yaml`'s
`SFTConfig(completion_only_loss=True)` (wired in the trainer factory, step 10) uses that
explicit split to find the loss boundary from **tokenized length**, not from matching a literal
string against the rendered sequence. There is no `response_template` left to keep in sync with
a chat template's exact token boundaries.

**This cell reproduces that exact boundary-finding logic by hand** on one real example, then --
this is the part the previous version of this cell did NOT do strongly enough -- **decodes the
unmasked (completion) tokens and asserts they are EXACTLY the gold assistant JSON**, not merely
that "some tokens are unmasked." A check that only confirms a field exists would not have caught
the `assistant_only_loss` trap above (that mechanism DOES return a mask -- it's just all zeros,
or in a different failure mode could be a valid-looking mask that's still wrong). Checking the
actual decoded content is the only check that can't be fooled that way.

In [ ]:
import json

example = raw_datasets["train"][0]
prompt_messages = example["prompt"]
completion_messages = example["completion"]

# This verifies the prompt/completion boundary is well-defined and exact at the TOKENIZER
# level: the rendered prompt text is a true prefix of the rendered full text (checked below),
# so tokenizing each separately still lines up for this tokenizer/template. It does NOT by
# itself prove this is what TRL's SFTTrainer does internally -- that is what the new cell
# right after the trainer factory (step 10) checks instead, by pulling a real batch from
# trainer.get_train_dataloader() and inspecting TRL's own labels directly. The two cells are
# complementary, not duplicates.
#
# An earlier version of this cell tokenized the prompt and the full prompt+completion
# separately with apply_chat_template(..., tokenize=True) and asserted the token ids lined up
# as a prefix -- that failed on a real Colab run against this tokenizer (the two token
# sequences did not line up). Fixed by rendering to TEXT first (tokenize=False), asserting the
# TEXT prefix relation (guaranteed by the chat template's own construction), and only then
# tokenizing each rendered string with add_special_tokens=False.
prompt_text = tokenizer.apply_chat_template(
    prompt_messages, tokenize=False, add_generation_prompt=True,
)
full_text = tokenizer.apply_chat_template(
    prompt_messages + completion_messages, tokenize=False, add_generation_prompt=False,
)
assert full_text.startswith(prompt_text), (
    "The rendered prompt text is not a prefix of the full rendered text."
)
full_ids = tokenizer(
    full_text, truncation=True, max_length=config["max_seq_len"],
    add_special_tokens=False,
)["input_ids"]
prompt_ids = tokenizer(
    prompt_text, truncation=True, max_length=config["max_seq_len"],
    add_special_tokens=False,
)["input_ids"]

n_masked = len(prompt_ids)
n_loss = len(full_ids) - n_masked
print(f"Example 0: {len(full_ids)} tokens total, {n_masked} masked (prompt, label=-100), "
      f"{n_loss} contribute to loss (completion).\n")

print(f"{'idx':>4} {'token_id':>8}  {'label':>8}  token_text")
for i, tid in enumerate(full_ids):
    is_completion = i >= n_masked
    label = tid if is_completion else -100
    tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
    marker = "LOSS  " if is_completion else "MASKED"
    print(f"{i:>4} {tid:>8}  {str(label):>8}  [{marker}] {tok_str!r}")

# CONTENT check -- not just "n_loss > 0". Decode the completion tokens and confirm they are
# EXACTLY the gold assistant JSON that was written to the JSONL. A check that only confirms
# some tokens are unmasked would NOT have caught the assistant_only_loss trap (that mechanism
# also "returns a mask" -- it's just all zeros); decoding and comparing real content is the
# check that can't be fooled that way.
decoded_completion = tokenizer.decode(full_ids[n_masked:], skip_special_tokens=True).strip()
gold_completion = completion_messages[0]["content"]
print(f"\nDecoded completion tokens (skip_special_tokens=True): {decoded_completion!r}")
print(f"Gold assistant content from the dataset:               {gold_completion!r}")

assert n_loss > 0, (
    "No completion tokens found -- STOP: do not train until this is fixed, or every "
    "example's loss would be masked entirely."
)
assert json.loads(decoded_completion) == json.loads(gold_completion), (
    "Decoded completion tokens do NOT match the gold assistant JSON -- the masking boundary "
    "is wrong (either cutting into the completion or including trailing text past it). "
    "STOP: do not train until this is fixed."
)
print("\nContent check passed: decoded completion tokens exactly match the gold assistant JSON.")

## 8. Confirm padding: dynamic per-batch, not fixed to `max_seq_len`

`max_seq_len` in `train/config.yaml` is a **truncation cap**, not a padding target. Verified by
hand: `tokenizer.pad()` pads each batch to **that batch's own longest sequence**, not to a fixed
length -- this is a property of the tokenizer/collator's padding call itself, independent of
which masking mechanism is used (step 7), so it no longer needs a trl collator object at all
(there isn't one to build any more -- see step 7). The cell below tokenizes real examples in the
new prompt-completion format (same way step 7 does) and pads two differently-sized batches.

In [ ]:
def tokenize_full(ex):
    text = tokenizer.apply_chat_template(
        ex["prompt"] + ex["completion"], tokenize=False, add_generation_prompt=False,
    )
    ids = tokenizer(
        text, truncation=True, max_length=config["max_seq_len"],
        add_special_tokens=False,
    )["input_ids"]
    return {"input_ids": ids}

tokenized_train = raw_datasets["train"].map(tokenize_full, remove_columns=raw_datasets["train"].column_names)

batch_a = [tokenized_train[i] for i in range(4)]
batch_b = [tokenized_train[i] for i in range(4, 4 + 8)]  # a differently-sized batch

padded_a = tokenizer.pad({"input_ids": [ex["input_ids"] for ex in batch_a]}, return_tensors="pt")
padded_b = tokenizer.pad({"input_ids": [ex["input_ids"] for ex in batch_b]}, return_tensors="pt")

print("batch A native lengths:", [len(ex["input_ids"]) for ex in batch_a])
print("batch A padded shape:  ", tuple(padded_a["input_ids"].shape))
print()
print("batch B native lengths:", [len(ex["input_ids"]) for ex in batch_b])
print("batch B padded shape:  ", tuple(padded_b["input_ids"].shape))
print()
print(f"max_seq_len from config: {config['max_seq_len']}")
print("If padding were fixed to max_seq_len, BOTH batches would show that same second dimension.")
print("Instead each batch is padded independently to its own longest row -- dynamic per-batch padding, confirmed.")

## 9. Load base model in 4-bit (QLoRA) + attach LoRA

Also normalizes any `torch.bfloat16` parameter to `COMPUTE_DTYPE` right after LoRA is
attached, with a printed before/after census and a hard, named assertion -- see the cell's
own comment for why `dtype=COMPUTE_DTYPE` on `from_pretrained` alone was not enough on a real
T4 run, and why this same check runs a second time inside `build_trainer` (step 10).

In [ ]:
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config["quantization"]["load_in_4bit"],
    bnb_4bit_quant_type=config["quantization"]["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # decided at runtime in cell 1 (GPU check), not from config.yaml
    bnb_4bit_use_double_quant=config["quantization"]["bnb_4bit_use_double_quant"],
)

model = AutoModelForCausalLM.from_pretrained(
    config["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    # WITHOUT this, the model's dtype comes from Qwen2.5's own config.json, which declares
    # bfloat16 -- verified on a real T4 Colab run: all 392 trainable LoRA parameters loaded as
    # torch.bfloat16. On a T4 the trainer runs fp16 mixed precision (USE_BF16 is False), and
    # fp16's GradScaler cannot unscale bfloat16 gradients:
    #     NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not
    #     implemented for 'BFloat16'
    # bnb_4bit_compute_dtype above does NOT cover this. That one only sets the dtype the 4-bit
    # base weights are dequantized to inside matmuls; the dtype of the trainable LoRA
    # parameters -- the ones that actually carry gradients -- is set here.
    # In transformers 5.x the kwarg is `dtype`; `torch_dtype` is deprecated.
    dtype=COMPUTE_DTYPE,
)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=config["gradient_checkpointing"]
)

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    bias=config["lora"]["bias"],
    task_type=config["lora"]["task_type"],
    target_modules=config["lora"]["target_modules"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- Self-verifying dtype normalization, not a single-cause guess -----------------------------
# The `dtype=COMPUTE_DTYPE` fix above did NOT resolve the crash on a real T4 run -- the
# identical NotImplementedError still occurred. Rather than guess a second single cause, this
# is a runtime-checked normalization: cast every torch.bfloat16 parameter to COMPUTE_DTYPE,
# print a before/after census, and hard-assert (by NAME, not just count) that none remain.
#
# This same function is called again inside build_trainer(), on the model the constructed
# SFTTrainer actually holds -- not redundant: reading trl 1.13.0's installed source
# (trainer/sft_trainer.py) found that SFTTrainer.__init__ unconditionally does, for any
# quantized model:
#     for param in model.parameters():
#         if param.requires_grad:
#             param.data = param.data.to(torch.bfloat16)
# citing the QLoRA paper's recommendation -- by design in trl, not a bug there, and it runs
# regardless of bnb_4bit_compute_dtype, regardless of args.bf16/args.fp16, and regardless of
# whether the model was already a PeftModel with no peft_config passed (this notebook's case,
# since LoRA is attached here, before SFTTrainer is ever constructed). That means THIS cell's
# normalization can pass cleanly and still be silently undone the moment build_trainer()
# constructs the trainer -- which is exactly why the same check runs again there. See
# PROGRESS.md's step-12 entry for the full source trace.
def normalize_bfloat16_params(m, target_dtype, label):
    before = Counter(str(p.dtype) for p in m.parameters())
    print(f"[{label}] dtype census BEFORE normalization: {dict(before)}")
    n_cast = 0
    for name, p in m.named_parameters():
        if p.dtype == torch.bfloat16:
            p.data = p.data.to(target_dtype)
            n_cast += 1
    after = Counter(str(p.dtype) for p in m.parameters())
    print(f"[{label}] dtype census AFTER normalization ({n_cast} param(s) cast): {dict(after)}")
    offenders = [name for name, p in m.named_parameters() if p.dtype == torch.bfloat16]
    assert not offenders, (
        f"STOP: do not train until this is fixed -- [{label}] {len(offenders)} parameter(s) "
        f"are still torch.bfloat16 after normalization: {offenders}"
    )
    return after

normalize_bfloat16_params(model, COMPUTE_DTYPE, "model-loading cell")

## 10. Trainer factory

One function, reused for both the smoke test and the full run, so the two runs differ only in
`max_steps`/`num_train_epochs`/`eval_dataset` -- every other hyperparameter comes from
`train/config.yaml`. Four things worth calling out:
- `completion_only_loss=config["completion_only_loss"]` is what actually applies the masking
  verified in step 7 -- no `data_collator=` argument is passed at all (there is no collator
  object to pass; trl 1.13.0 exports none -- see step 7 for the full story). `SFTTrainer`
  determines the prompt/completion boundary itself from the dataset's `"prompt"`/`"completion"`
  columns.
- `load_best_model_at_end` is always `False` (see `train/config.yaml`'s "Evaluation &
  checkpointing" comments): during training, `eval_dataset` is the fast subsample, so Trainer's
  own best-checkpoint tracking would be picking against that subsample, not the full validation
  set -- final selection is done manually in step 14, over the full set.
- **trl 1.13.0's `SFTConfig` doesn't have `max_seq_length` or `warmup_ratio`** (probed against
  the real installed class on Colab -- 133 fields total). `max_seq_length` is a straight
  rename to `max_length`. `warmup_ratio` has no equivalent at all -- only `warmup_steps`
  (default 0) -- so this function converts `train/config.yaml`'s `warmup_ratio` into an
  absolute step count itself, computed fresh per call from that call's own total step count
  (the smoke test and the full run are very different lengths), and **prints the resolved
  value** before building `SFTConfig` so a wrong number can't silently survive a whole run.
- **Right before returning, the same `normalize_bfloat16_params` from step 9 runs again**,
  this time on `sft_trainer.model` -- the model the constructed `SFTTrainer` actually holds,
  not the one passed into it. Confirmed by reading trl 1.13.0's installed source:
  `SFTTrainer.__init__` unconditionally casts every trainable parameter to `bfloat16` for a
  quantized model, undoing step 9's normalization. `sft_config.fp16`/`sft_config.bf16` and
  the trainer's `accelerator.mixed_precision` are also printed here, so an independent bf16
  selection by `accelerate` (a different bug) would be visible too.

In [ ]:
from trl import SFTConfig, SFTTrainer
import math

def build_trainer(output_dir, eval_dataset, max_steps=None, num_train_epochs=None, eval_steps=None, logging_steps=None):
    resolved_num_train_epochs = num_train_epochs if num_train_epochs is not None else config["num_train_epochs"]

    # trl 1.13.0's SFTConfig dropped warmup_ratio (probed against the real installed class on
    # Colab: 133 fields total, none containing "ratio" -- only warmup_steps, default 0). The
    # human-meaningful warmup_ratio stays in train/config.yaml; converted to an absolute step
    # count here instead, per call, because the smoke test (max_steps=100) and the full run
    # (~2,450 steps) are very different lengths -- a single fixed warmup_steps would be wrong
    # for one of them.
    if max_steps is not None:
        total_steps = max_steps
    else:
        effective_batch = config["per_device_train_batch_size"] * config["gradient_accumulation_steps"]
        steps_per_epoch = math.ceil(len(raw_datasets["train"]) / effective_batch)
        total_steps = steps_per_epoch * resolved_num_train_epochs

    warmup_ratio = config["warmup_ratio"]
    warmup_steps = max(1, round(warmup_ratio * total_steps)) if warmup_ratio > 0 else 0
    print(f"warmup: ratio {warmup_ratio} x {total_steps} total steps -> warmup_steps={warmup_steps}")

    # disable_mixed_precision (train/config.yaml) is a LAST-RESORT fallback, not the intended
    # path -- see the config comment. When true, both flags below go False, so no GradScaler
    # is ever constructed and training runs in plain fp32 (slower, more memory, may OOM on a
    # T4's 15GB).
    mixed_precision_disabled = config["disable_mixed_precision"]
    if mixed_precision_disabled:
        print("disable_mixed_precision is TRUE in train/config.yaml -- training in fp32, no GradScaler.")
    resolved_bf16 = USE_BF16 and not mixed_precision_disabled
    resolved_fp16 = (not USE_BF16) and not mixed_precision_disabled

    sft_config = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=config["per_device_train_batch_size"],
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=float(config["learning_rate"]),
        lr_scheduler_type=config["lr_scheduler_type"],
        warmup_steps=warmup_steps,  # trl 1.13.0's SFTConfig has no warmup_ratio -- converted above
        weight_decay=config["weight_decay"],
        optim=config["optim"],
        max_grad_norm=config["max_grad_norm"],
        num_train_epochs=resolved_num_train_epochs,
        max_steps=max_steps if max_steps is not None else -1,
        eval_strategy=config["eval_strategy"],
        eval_steps=eval_steps if eval_steps is not None else config["eval_steps"],
        save_strategy=config["save_strategy"],
        save_steps=eval_steps if eval_steps is not None else config["save_steps"],
        logging_steps=logging_steps if logging_steps is not None else config["logging_steps"],
        save_total_limit=config["save_total_limit"],
        load_best_model_at_end=config["load_best_model_at_end"],  # always False -- see cell 9's note
        metric_for_best_model=config["metric_for_best_model"],
        greater_is_better=config["greater_is_better"],
        max_length=config["max_seq_len"],  # trl 1.13.0's SFTConfig has no max_seq_length -- renamed
        completion_only_loss=config["completion_only_loss"],  # masks the prompt via the dataset's own prompt/completion split -- see step 7
        seed=config["seed"],
        bf16=resolved_bf16,   # decided at runtime in cell 1 (GPU check) -- False on Turing (e.g. T4) -- and by disable_mixed_precision above
        fp16=resolved_fp16,
        report_to=[],
        push_to_hub=False,  # pushed manually later, after training finishes and the best checkpoint is loaded
    )
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=raw_datasets["train"],
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    # --- Re-check + re-normalize the model the trainer ACTUALLY holds, right before returning --
    # "This is the point that matters" -- the model-loading cell's normalization (step 9) can
    # pass cleanly and still be wrong here, because SFTTrainer.__init__ (trl 1.13.0) has its
    # own unconditional bfloat16 cast for quantized models -- confirmed by reading trl's
    # installed source (see step 9's comment and PROGRESS.md's step-12 entry for the exact
    # code found). Beyond a literal "check only" at this point: since that cast is known to
    # fire every time for this notebook's quantized model, this deliberately calls the SAME
    # normalize_bfloat16_params (cast + census + named assert) again here, not just an
    # assert -- otherwise this check would fail on every single run with no way to proceed.
    print(f"sft_config.fp16={sft_config.fp16}  sft_config.bf16={sft_config.bf16}  "
          f"trainer.accelerator.mixed_precision={getattr(sft_trainer.accelerator, 'mixed_precision', '<unknown>')!r}")
    normalize_bfloat16_params(sft_trainer.model, COMPUTE_DTYPE, "build_trainer (post-SFTTrainer construction)")

    return sft_trainer

## Verify TRL's actual labels (not our own reconstruction)

Every check so far (step 7) is OUR OWN reconstruction of where the prompt/completion boundary
should be, built by hand outside of `SFTTrainer` -- nothing yet has looked at the actual
tensors TRL feeds the model during training. This cell closes that gap empirically: it builds
a throwaway trainer via `build_trainer()`, pulls one real batch from
`trainer.get_train_dataloader()`, and checks TRL's own `labels` tensor directly.

**This is the check that cannot be fooled the way `assistant_only_loss` fooled us earlier** --
that mechanism also "returned a mask," it was just silently all zeros, with no error. This
cell decodes the actual unmasked (`label != -100`) token ids TRL produced for every row in the
batch and asserts each one parses as valid JSON with exactly the keys `queue`, `priority`,
`type` -- a structural, content-level check on TRL's real output, not a boundary computed
independently of it (which is what step 7 verifies instead -- a different, complementary
check, not a duplicate of this one).

**Not executable/testable locally** (no GPU here -- same constraint noted throughout this
notebook and in `PROGRESS.md`'s step-07 entries).

In [ ]:
import json
import tempfile

label_check_dir = tempfile.mkdtemp(prefix="trl_label_check_")
label_check_trainer = build_trainer(output_dir=label_check_dir, eval_dataset=eval_subsample)

batch = next(iter(label_check_trainer.get_train_dataloader()))
batch_labels = batch["labels"]
batch_input_ids = batch["input_ids"]

print(f"Batch size: {batch_labels.shape[0]}, sequence length: {batch_labels.shape[1]}")

for row in range(batch_labels.shape[0]):
    row_labels = batch_labels[row]
    row_input_ids = batch_input_ids[row]
    unmasked_positions = (row_labels != -100).nonzero(as_tuple=True)[0]
    n_masked = int((row_labels == -100).sum())
    n_unmasked = int(unmasked_positions.numel())

    assert n_unmasked > 0, (
        f"STOP: do not train until this is fixed -- row {row} has ZERO unmasked label "
        "positions (every token in this row would be masked from the loss)."
    )

    unmasked_ids = row_input_ids[unmasked_positions].tolist()
    decoded = tokenizer.decode(unmasked_ids, skip_special_tokens=True).strip()

    try:
        parsed = json.loads(decoded)
    except json.JSONDecodeError as e:
        raise AssertionError(
            f"STOP: do not train until this is fixed -- row {row}'s unmasked label tokens do "
            f"not decode to valid JSON: {e}. Decoded text: {decoded!r}"
        ) from e

    assert set(parsed.keys()) == {"queue", "priority", "type"}, (
        f"STOP: do not train until this is fixed -- row {row}'s decoded JSON has keys "
        f"{sorted(parsed.keys())}, expected exactly ['priority', 'queue', 'type']. "
        f"Decoded text: {decoded!r}"
    )

    if row == 0:
        print(f"\nRow 0: {n_masked} masked, {n_unmasked} unmasked (label != -100) positions.")
        print(f"Row 0 decoded unmasked text: {decoded!r}")

print(f"\nAll {batch_labels.shape[0]} rows in this batch: TRL's own unmasked labels decode to "
      "valid JSON with exactly the keys queue/priority/type.")

## 11. Smoke test (~100 steps)

Confirms the loss is actually falling and the model produces valid JSON **before** committing
to the full multi-epoch run. Uses `smoke_test.*` from `train/config.yaml`, evaluated against the
subsample (speed only, same as the full run below).

In [ ]:
smoke_output_dir = f"{REPO_ROOT}/{config['smoke_test_output_dir']}"
smoke_trainer = build_trainer(
    output_dir=smoke_output_dir,
    eval_dataset=eval_subsample,
    max_steps=config["smoke_test"]["max_steps"],
    eval_steps=config["smoke_test"]["eval_steps"],
    logging_steps=config["smoke_test"]["logging_steps"],
)
smoke_result = smoke_trainer.train()
print(smoke_result)

# --- Checkpoint disk-size check (LoRA adapters only, not full model weights) ------------------
# save_total_limit is null (train/config.yaml) -- EVERY checkpoint from the full run is kept,
# on purpose (see train/config.yaml's comment: with load_best_model_at_end=False, a finite
# limit would silently delete the true best checkpoint before step 14 gets a chance to find
# it). That only works if Colab's disk can actually hold all of them -- checked HERE, before
# the full run, using a real saved checkpoint from this smoke test, not a guess.
import glob, math, shutil
from pathlib import Path

def dir_size_bytes(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())

smoke_ckpt_dirs = sorted(
    glob.glob(f"{smoke_output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1])
)
assert smoke_ckpt_dirs, f"No checkpoint-* directories found under {smoke_output_dir} -- smoke test didn't save any."
one_ckpt_dir = smoke_ckpt_dirs[-1]
one_ckpt_mb = dir_size_bytes(one_ckpt_dir) / (1024 ** 2)
print(f"\nOne saved checkpoint ({one_ckpt_dir}): {one_ckpt_mb:.1f} MB on disk "
      "(LoRA adapter only -- the 4-bit base model is never re-saved).")

# Projected checkpoint count for the FULL run -- same math as train/config.yaml's comment.
effective_batch_size = config["per_device_train_batch_size"] * config["gradient_accumulation_steps"]
steps_per_epoch = math.ceil(len(raw_datasets["train"]) / effective_batch_size)
full_total_steps = steps_per_epoch * config["num_train_epochs"]
full_num_checkpoints = math.ceil(full_total_steps / config["save_steps"])
projected_total_gb = (one_ckpt_mb * full_num_checkpoints) / 1024
print(f"Full run: {full_total_steps} steps ({steps_per_epoch}/epoch x {config['num_train_epochs']} epochs), "
      f"save_steps={config['save_steps']} -> ~{full_num_checkpoints} checkpoints "
      f"(save_total_limit is null, so all of them survive).")
print(f"Projected total disk for ALL full-run checkpoints: ~{projected_total_gb:.2f} GB")

# REPO_ROOT now defaults to a Google Drive-mounted FUSE path (see cell 7), not local disk.
# Print the raw numbers shutil.disk_usage() actually returns for it -- it is NOT verified
# whether this reports Google Drive's real account quota on a FUSE-mounted path, or something
# else entirely (e.g. the underlying Colab VM's own local filesystem size, which would be
# meaningless for a Drive path). Not assumed either way; printed raw so it can be judged by
# eye against what drive.google.com actually shows for this account.
disk_usage = shutil.disk_usage(REPO_ROOT)
total_gb = disk_usage.total / (1024 ** 3)
used_gb = disk_usage.used / (1024 ** 3)
free_gb = disk_usage.free / (1024 ** 3)
print(f"\nshutil.disk_usage({REPO_ROOT!r}) raw result -- total: {total_gb:.2f} GB, "
      f"used: {used_gb:.2f} GB, free: {free_gb:.2f} GB")
print(
    "NOTE: if REPO_ROOT is a Google Drive path (the default -- see cell 7), it is UNVERIFIED "
    "whether the numbers above reflect this Drive account's real quota. If they look "
    "implausible for your account (e.g. far larger/smaller than what drive.google.com shows "
    "under Storage), verify free space by hand there before trusting the projection below."
)

if projected_total_gb > free_gb:
    print(
        "\n*** WARNING: projected checkpoint total EXCEEDS free disk space at REPO_ROOT. ***\n"
        "Do NOT proceed to the full training run as configured -- this is a real problem, not "
        "a formality. Report this back before continuing so a fix (a smaller save_total_limit, "
        "a different REPO_ROOT/output_dir, clearing space, etc.) can be decided together -- "
        "this notebook will NOT silently reduce save_total_limit on its own."
    )
else:
    print("Free disk space looks sufficient for all full-run checkpoints (LoRA adapters only) "
          "-- but see the NOTE above if REPO_ROOT is on Google Drive and this number looks off.")

### Smoke-test loss curve

In [ ]:
import matplotlib.pyplot as plt

history = smoke_trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss", marker="o")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label=f"eval loss ({len(eval_subsample)}-example subsample)", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Smoke test loss (should trend downward)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 5 generated outputs on val examples (smoke-test model)

In [ ]:
import json
from ticket_router.schema import TicketLabels
from pydantic import ValidationError

def generate_one(model, prompt_messages, max_new_tokens=64):
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            prompt_ids, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    completion_ids = out_ids[0][prompt_ids.shape[1]:]
    return tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

n_show = config["smoke_test"]["n_generated_examples"]
for i in range(n_show):
    example = raw_datasets["val"][i]
    generated = generate_one(smoke_trainer.model, example["prompt"])
    gold = example["completion"][0]["content"]
    try:
        TicketLabels(**json.loads(generated))
        status = "VALID JSON, matches schema"
    except (json.JSONDecodeError, ValidationError) as e:
        status = f"INVALID: {e}"
    print(f"--- val[{i}] ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}")
    print(f"  status:    {status}\n")

## 12. STOP AND CONFIRM before continuing

**Do not run the cells below until you've checked, above:**
- The smoke-test loss curve is trending **downward** (not flat, not increasing).
- At least most of the 5 generated outputs are **valid JSON matching the 3-key schema**
  (`queue`, `priority`, `type`) -- they do not need to be *correct* yet at only ~100 steps,
  just structurally valid.
- The checkpoint disk-size projection printed at the end of the smoke-test cell did **not**
  print a `WARNING`. If it did, stop here -- `save_total_limit` is null so every full-run
  checkpoint is kept, and this is where a real disk shortfall would be caught before it
  happens partway through the full run.

If any of these look wrong, stop here and investigate (check `train/config.yaml`, the masking
cell's content check, and the model/trainer setup above) before spending a full ~45-90 minute
run on it. Everything below this cell is the **full, multi-epoch training run**.

## 13. Full training run

Mid-training evaluation (every `eval_steps`=250) runs against the `eval_subsample` -- speed
only. The full validation set is used once, in step 14 below, to pick the actual best
checkpoint.

In [ ]:
import glob

full_output_dir = f"{REPO_ROOT}/{config['output_dir']}"
trainer = build_trainer(output_dir=full_output_dir, eval_dataset=eval_subsample)

# Resume automatically if a checkpoint from a previous (possibly disconnected) session already
# exists under full_output_dir -- do NOT pass resume_from_checkpoint=True unconditionally,
# since Trainer raises if the directory has no checkpoint at all (e.g. the very first run).
# REPO_ROOT defaults to Google Drive (see cell 7) specifically so these checkpoints survive a
# Colab disconnect for this to actually have something to resume from.
existing_checkpoints = sorted(
    glob.glob(f"{full_output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1])
)
if existing_checkpoints:
    resume_from = existing_checkpoints[-1]
    print(f"RESUMING from existing checkpoint: {resume_from}")
    train_result = trainer.train(resume_from_checkpoint=resume_from)
else:
    print(f"No existing checkpoint found under {full_output_dir} -- starting a fresh training run.")
    train_result = trainer.train()
print(train_result)

### Full-run loss curve

In [ ]:
history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label=f"eval loss ({len(eval_subsample)}-example subsample)", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Full run: train loss vs. SUBSAMPLE validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(
    f"NOTE: the eval loss above is against the {len(eval_subsample)}-example subsample (speed "
    "only) -- it is NOT the full-validation metric and is NOT used to pick the final checkpoint. "
    "load_best_model_at_end is False for exactly this reason; the next cell selects the best "
    f"checkpoint using all {len(raw_datasets['val'])} full validation examples instead."
)
print(f"Checkpoints saved during this run (save_steps={config['save_steps']}, "
      f"save_total_limit={config['save_total_limit']}) are on disk under: {full_output_dir}")

## 14. Select the best checkpoint using the FULL validation set

Per `train/config.yaml`: mid-training evaluation used the `eval_subsample` for speed, so it
cannot tell us which of the checkpoints saved during training is actually best. `save_total_limit`
is null (see `train/config.yaml`), so every checkpoint written during the full run is still on
disk -- none were evicted by Trainer's rotation, which is exactly why this step can look at all
of them instead of only whichever ones happened to survive. This cell re-evaluates every
checkpoint saved under `output_dir` on the **full** 1,635-example validation set, by
`metric_for_best_model` (`eval_loss`, `greater_is_better`=False), and loads the winner into
`trainer.model` so every cell below (save, push, preview) uses it.

**Not executable/testable locally** (no GPU here -- same constraint noted at the top of this
notebook and in `PROGRESS.md`'s step-07 entry). Written to the current standard
`transformers`/`trl`/`peft` checkpoint-loading API; report back if anything needs correcting.

In [ ]:
import glob
from peft import PeftModel

def evaluate_checkpoint_on_full_val(checkpoint_dir):
    """Load one saved LoRA checkpoint fresh and evaluate it on the FULL validation set."""
    base = AutoModelForCausalLM.from_pretrained(
        config["base_model"], quantization_config=bnb_config, device_map="auto",
        dtype=COMPUTE_DTYPE,  # same reason as step 9 -- without it Qwen2.5's config.json forces bfloat16
    )
    ckpt_model = PeftModel.from_pretrained(base, checkpoint_dir)
    eval_only_config = SFTConfig(
        output_dir=f"{checkpoint_dir}/_full_val_eval_tmp",
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        max_length=config["max_seq_len"],  # trl 1.13.0's SFTConfig has no max_seq_length -- renamed
        completion_only_loss=config["completion_only_loss"],  # match training's loss definition -- see step 7/10
        seed=config["seed"],
        bf16=USE_BF16,
        fp16=not USE_BF16,
        report_to=[],
    )
    eval_trainer = SFTTrainer(
        model=ckpt_model,
        args=eval_only_config,
        # trl 1.13.0's SFTTrainer.__init__ raises ValueError: `train_dataset` is required --
        # older trl allowed an eval-only trainer, 1.13.0 does not. .evaluate() below never
        # iterates train_dataset, so one row satisfies the requirement without the cost of
        # trl processing the full 13,070-row train split once per checkpoint (10 times over
        # the run) for absolutely nothing.
        train_dataset=raw_datasets["train"].select(range(1)),
        eval_dataset=raw_datasets["val"],  # the FULL validation set, on purpose
        processing_class=tokenizer,
    )
    # Same unconditional bfloat16 cast as during training (trl 1.13.0's SFTTrainer.__init__,
    # for any quantized model -- see step 10's comment and PROGRESS.md's step-12 entry for the
    # source trace) happens here too. Without re-normalizing, this checkpoint's loss would be
    # computed at a different dtype than the training-time losses were, making the
    # cross-checkpoint eval_loss comparison below unreliable.
    normalize_bfloat16_params(eval_trainer.model, COMPUTE_DTYPE, f"evaluate_checkpoint_on_full_val({checkpoint_dir})")
    metrics = eval_trainer.evaluate()
    del base, ckpt_model, eval_trainer
    torch.cuda.empty_cache()
    return metrics["eval_loss"]

checkpoint_dirs = sorted(
    glob.glob(f"{full_output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1])
)
assert checkpoint_dirs, f"No checkpoint-* directories found under {full_output_dir} -- did training actually save any?"
print(f"Found {len(checkpoint_dirs)} saved checkpoint(s): {checkpoint_dirs}")

full_val_losses = {}
for ckpt_dir in checkpoint_dirs:
    loss = evaluate_checkpoint_on_full_val(ckpt_dir)
    full_val_losses[ckpt_dir] = loss
    print(f"  {ckpt_dir}: FULL-val eval_loss = {loss:.4f}  (n={len(raw_datasets['val'])})")

best_ckpt_dir = min(full_val_losses, key=full_val_losses.get)
print(f"\nBest checkpoint by FULL-val eval_loss: {best_ckpt_dir}  (eval_loss={full_val_losses[best_ckpt_dir]:.4f})")

# Load the winner into `trainer.model` -- every cell below (save/push/preview) now uses the
# checkpoint selected on the FULL validation set, not the subsample-monitored last step.
best_base = AutoModelForCausalLM.from_pretrained(
    config["base_model"], quantization_config=bnb_config, device_map="auto",
    dtype=COMPUTE_DTYPE,  # same reason as step 9 -- without it Qwen2.5's config.json forces bfloat16
)
trainer.model = PeftModel.from_pretrained(best_base, best_ckpt_dir)
print(f"trainer.model now holds the checkpoint from {best_ckpt_dir}.")

## 15. Save the adapter + push to the Hugging Face Hub

`trainer.model` is the checkpoint selected in step 14 by FULL-validation `eval_loss`. Requires
`hub_model_id` in `train/config.yaml` (already set to
`Omar-Mohsen22/ticket-router-qwen2.5-1.5b-qlora`) and a write-access HF token, already confirmed
present in step 5.

In [ ]:
adapter_dir = f"{REPO_ROOT}/{config['adapter_output_dir']}"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter + tokenizer saved to {adapter_dir}")

if config["push_to_hub"]:
    if not config["hub_model_id"]:
        raise ValueError(
            "push_to_hub is true in train/config.yaml but hub_model_id is empty. "
            "Set hub_model_id to '<your-hf-username>/<repo-name>' and re-run this cell."
        )
    # Reuses the token already validated (write access confirmed) in step 5 -- no interactive
    # login() prompt needed here.
    trainer.model.push_to_hub(config["hub_model_id"], private=config["hub_private"], token=token)
    tokenizer.push_to_hub(config["hub_model_id"], private=config["hub_private"], token=token)
    print(f"Pushed adapter to https://huggingface.co/{config['hub_model_id']}")
else:
    print("push_to_hub is false in train/config.yaml -- adapter saved locally only.")

## 16. Post-train preview: 20 val examples

A quick eyeball check of the trained model's output format and rough quality, right after
training and before running the project's full evaluation harness (`evaluate.py`) separately.
Uses `post_train_preview.n_val_examples` from `train/config.yaml`.

In [ ]:
n_preview = config["post_train_preview"]["n_val_examples"]
preview_rows = []
for i in range(n_preview):
    example = raw_datasets["val"][i]
    generated = generate_one(trainer.model, example["prompt"])
    gold = example["completion"][0]["content"]
    try:
        parsed = TicketLabels(**json.loads(generated))
        status = "valid"
    except (json.JSONDecodeError, ValidationError) as e:
        parsed, status = None, f"invalid: {e}"
    preview_rows.append((i, generated, gold, status))
    print(f"--- val[{i}] ({status}) ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}\n")

n_valid = sum(1 for _, _, _, s in preview_rows if s == "valid")
n_exact = sum(1 for _, g, gold, s in preview_rows if s == "valid" and g == gold)
print(f"{n_valid}/{n_preview} valid JSON, {n_exact}/{n_preview} exact string match on these {n_preview} examples.")
print("This is a quick eyeball check only -- run src/ticket_router/evaluate.py locally against "
      "a full predictions file for real macro-F1/accuracy numbers on the harness.")